# Verifying `train_lora.py`'s assumptions about the model's layer structure

This notebook does **not** reimplement the logic that decides which layers get a LoRA adapter — it imports the real functions from `train_lora.py` (`language_model_of`, `find_target_modules`, `resolve_image_token_id`, `assert_only_language_trainable`) and runs them against the actual model, so what you see here is exactly what the trainer will do, not a paraphrase of it. (`train_lora.py` trains **plain LoRA by default** — `--use_dora` is opt-in and not what this project actually trains with, since vLLM can't serve a DoRA adapter selectively; see that script's module docstring.)

**Run this on the cluster** (`conda activate tfm` — not a different env; an earlier run under a `cv` env failed to import `Gemma4UnifiedProcessor` because that env's `transformers` predates this model's support) — this machine has no `torch`/`transformers` installed, so it was written but not executed here.

## Why the model is built on the `meta` device

`google/gemma-4-12B-it` is a multi-GB checkpoint. Everything we need to check here — module names, module types, which Linear layers exist and where, parameter shapes/counts — is determined entirely by the model's **architecture** (from its config), not by the actual weight values. `accelerate.init_empty_weights()` builds the real `nn.Module` tree with real shapes but weights on the `meta` device (no data, no download, no memory). `named_modules()`, `named_parameters()`, `isinstance(...)` checks, and `.numel()` on shapes all work identically to a fully materialized model.

Plain LoRA's own init needs no data from the base weights, so even section 7's real-weight check would work on a meta model in principle — it loads real weights anyway so it exercises the literal code path `train_lora.py` runs, not just the parts that happen to tolerate a meta tensor.

In [19]:
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "fine_tuning" else os.getcwd()
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, "fine_tuning"))

import torch
from transformers import AutoConfig, AutoModelForImageTextToText, AutoProcessor

# The REAL functions this notebook is verifying — imported, not reimplemented.
from train_lora import (
    language_model_of,
    find_target_modules,
    assert_only_language_trainable,
)
from vision_cache import resolve_image_token_id

MODEL_ID = "google/gemma-4-12B-it"  # change to spot-check another family (qwen/mistral/llava/internvl)

## 1. Build the model on the `meta` device (architecture only, no weight download)

In [20]:
config = AutoConfig.from_pretrained(MODEL_ID)

try:
    from accelerate import init_empty_weights
    with init_empty_weights():
        model = AutoModelForImageTextToText.from_config(config)
    print("Built on the meta device via accelerate.init_empty_weights() — no weights downloaded.")
except ImportError:
    print("accelerate not available — falling back to a REAL weight load (slower, downloads the checkpoint).")
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="cpu")

print(type(model).__name__)

Built on the meta device via accelerate.init_empty_weights() — no weights downloaded.
Gemma4UnifiedForConditionalGeneration


## 2. Top-level module tree

The thing to eyeball here: how many top-level submodules the wrapper has, and which ones look like "vision" vs "language" vs "projector". This is the ground truth `language_model_of()` is trying to identify programmatically below.

In [21]:
def param_count(module):
    """Works on meta tensors too — shapes are real, only the data is absent."""
    return sum(p.numel() for p in module.parameters())

print(f"{'submodule':30s} {'type':40s} {'params':>15s}")
print("-" * 90)
for name, sub in model.named_children():
    print(f"{name:30s} {type(sub).__name__:40s} {param_count(sub):>15,}")
print("-" * 90)
print(f"{'TOTAL':30s} {'':40s} {param_count(model):>15,}")

submodule                      type                                              params
------------------------------------------------------------------------------------------
model                          Gemma4UnifiedModel                        11,959,730,176
lm_head                        Linear                                     1,006,632,960
------------------------------------------------------------------------------------------
TOTAL                                                                    12,966,363,136


## 3. `language_model_of(model)` — which submodule `train_lora.py` treats as the decoder

**This check caught a real bug.** An earlier version of `language_model_of` only looked at TOP-LEVEL attributes (`model.language_model`, falling back to `model.model`). Run against the actual `google/gemma-4-12B-it` tree, that resolved to `model.model` — `Gemma4UnifiedModel` — which is NOT the decoder, it's the wrapper holding THREE siblings: `language_model` (the real decoder), `embed_vision`, and `embed_audio` (this is an omni-modal checkpoint, text+image+audio, not just text+image). Every Linear inside `embed_vision`/`embed_audio` would have been targeted for adapters right alongside the real decoder — invisible from the loss curve, since the bug only widens *which* weights get adapted, not whether training runs.

`language_model_of` now searches the WHOLE module tree for a submodule literally named `language_model` and takes the SHALLOWEST match, so it's correct whether an architecture nests it at depth 1 (Qwen-VL/LLaVA-style: `model.language_model`) or deeper (this checkpoint: `model.model.language_model`). Confirm below that the resolved module actually looks like a decoder (embedding table, a stack of transformer `layers`, a final norm), not a wrapper with encoder siblings.

In [22]:
lm, lm_prefix = language_model_of(model)
print(f"Resolved attribute: model.{lm_prefix}  ->  {type(lm).__name__}")
print(f"Parameters under it: {param_count(lm):,} / {param_count(model):,} "
      f"({100 * param_count(lm) / param_count(model):.1f}% of the whole model)\n")
print("Its own immediate children (should read like a decoder: embed_tokens, layers, norm, ...):")
for name, sub in lm.named_children():
    print(f"  {name:20s} {type(sub).__name__}")

Resolved attribute: model.model.language_model  ->  Gemma4UnifiedTextModel
Parameters under it: 11,907,350,272 / 12,966,363,136 (91.8% of the whole model)

Its own immediate children (should read like a decoder: embed_tokens, layers, norm, ...):
  embed_tokens         Gemma4UnifiedTextScaledWordEmbedding
  layers               ModuleList
  norm                 Gemma4UnifiedRMSNorm
  rotary_emb           Gemma4UnifiedTextRotaryEmbedding


In [23]:

# Regression guard for the exact bug this section documents: the resolved
# decoder must not itself contain another encoder as a child (which is what
# happened when language_model_of used to stop at model.model).
_suspect_names = ("vision", "audio", "image", "visual", "encoder")
_suspect_children = [name for name, _ in lm.named_children()
                     if any(s in name.lower() for s in _suspect_names)]
assert not _suspect_children, (
    f"The resolved decoder still has encoder-looking children: {_suspect_children} — "
    f"language_model_of() likely stopped one level too shallow again. Investigate before training."
)
print("OK: no vision/audio/encoder-looking submodule inside the resolved decoder.")


OK: no vision/audio/encoder-looking submodule inside the resolved decoder.


### Sanity check: does a `layers` stack exist, and does one layer look like a standard decoder block?

In [24]:
layers = getattr(lm, "layers", None)
assert layers is not None, "Expected a `.layers` stack on the language decoder — inspect the tree above manually."
print(f"{len(layers)} decoder layers. Layer 0's structure:\n")
for name, sub in layers[0].named_modules():
    if name:
        print(f"  {name:30s} {type(sub).__name__}")

48 decoder layers. Layer 0's structure:

  self_attn                      Gemma4UnifiedTextAttention
  self_attn.q_proj               Linear
  self_attn.q_norm               Gemma4UnifiedRMSNorm
  self_attn.k_norm               Gemma4UnifiedRMSNorm
  self_attn.v_norm               Gemma4UnifiedRMSNorm
  self_attn.k_proj               Linear
  self_attn.v_proj               Linear
  self_attn.o_proj               Linear
  mlp                            Gemma4UnifiedTextMLP
  mlp.gate_proj                  Linear
  mlp.up_proj                    Linear
  mlp.down_proj                  Linear
  mlp.act_fn                     GELUTanh
  input_layernorm                Gemma4UnifiedRMSNorm
  post_attention_layernorm       Gemma4UnifiedRMSNorm
  pre_feedforward_layernorm      Gemma4UnifiedRMSNorm
  post_feedforward_layernorm     Gemma4UnifiedRMSNorm


## 4. `find_target_modules(model, lm_prefix)` — which Linear layers actually get a DoRA adapter

Matched by **full dotted name**, not the usual suffix shorthand (`"q_proj"`) — that distinction is the whole point of this check. If any Linear layer *outside* the decoder happens to share a suffix with one inside it (very common: most ViT attention blocks also have `q_proj`/`v_proj`), a suffix-based match would silently adapt the vision tower too. The cell below makes that risk visible: it lists every Linear layer in the WHOLE model, then shows which ones were selected vs. excluded — by full name, not grouped by top-level submodule, since on an architecture like this one every encoder lives under the SAME top-level `model` attribute as the decoder (see section 3), so a top-level grouping can't actually distinguish them.

In [25]:
targets = find_target_modules(model, lm_prefix)
print(f"{len(targets)} Linear modules targeted for a LoRA adapter, all under 'model.{lm_prefix}.'\n")
print("First 5:", targets[:5])
print("Last 5: ", targets[-5:])

328 Linear modules targeted for a LoRA adapter, all under 'model.model.language_model.'

First 5: ['model.language_model.layers.0.self_attn.q_proj', 'model.language_model.layers.0.self_attn.k_proj', 'model.language_model.layers.0.self_attn.v_proj', 'model.language_model.layers.0.self_attn.o_proj', 'model.language_model.layers.0.mlp.gate_proj']
Last 5:  ['model.language_model.layers.47.self_attn.k_proj', 'model.language_model.layers.47.self_attn.o_proj', 'model.language_model.layers.47.mlp.gate_proj', 'model.language_model.layers.47.mlp.up_proj', 'model.language_model.layers.47.mlp.down_proj']


In [26]:
all_linears = {name for name, mod in model.named_modules() if isinstance(mod, torch.nn.Linear)}
targeted = set(targets)
excluded = sorted(all_linears - targeted)

print(f"Linear layers in the WHOLE model: {len(all_linears)}")
print(f"Targeted (get a LoRA adapter):    {len(targeted)}")
print(f"Excluded (frozen, no adapter):    {len(excluded)}\n")

# NOTE: grouping excluded names by their first "." segment (e.g. Counter over
# name.split(".", 1)[0]) is USELESS on an architecture like this one, where
# language_model/embed_vision/embed_audio are all nested under the SAME
# top-level "model" attribute — every excluded Linear would show up under the
# single bucket "model", telling you nothing about which encoder it's
# actually in. With `excluded` this short, just print every name directly.
print("Every excluded Linear (should be entirely vision tower / audio encoder /\n"
      "multimodal projector / lm_head — NOTHING starting with the decoder's own path):")
for name in excluded:
    flag = "  <-- INSIDE THE DECODER, THIS IS A BUG" if name.startswith(lm_prefix + ".") else ""
    print(f"  {name}{flag}")

assert not any(name.startswith(lm_prefix + ".") for name in excluded), (
    "A Linear layer INSIDE the decoder was excluded from targeting — investigate before training."
)
assert all(name.startswith(lm_prefix + ".") for name in targeted), (
    "A targeted Linear layer is OUTSIDE the decoder — this is exactly the bug the full-name "
    "matching in find_target_modules() is meant to prevent. Investigate before training."
)
print("\nOK: targeted == every decoder Linear, excluded == every non-decoder Linear.")

Linear layers in the WHOLE model: 332
Targeted (get a LoRA adapter):    328
Excluded (frozen, no adapter):    4

Every excluded Linear (should be entirely vision tower / audio encoder /
multimodal projector / lm_head — NOTHING starting with the decoder's own path):
  lm_head
  model.embed_audio.embedding_projection
  model.embed_vision.multimodal_embedder.embedding_projection
  model.embed_vision.patch_dense

OK: targeted == every decoder Linear, excluded == every non-decoder Linear.


## 5. `resolve_image_token_id(model, processor)` — where image features get scattered into `input_ids`

This id is what `vision_cache.inject_image_features` uses to place the cached embeddings. A wrong value here doesn't crash — it scatters features into the wrong sequence positions and trains happily on nonsense, which is exactly why `train_lora.py --verify_cache` exists as a second, independent check at run time. This cell is the first one.

**If `AutoProcessor.from_pretrained` fails with `ModuleNotFoundError: Could not import module '...Processor'`**: that message is transformers' lazy-loader wrapping WHATEVER exception the module actually threw on import — it does not necessarily mean a version mismatch. This checkpoint is **omni-modal** (text+image+**audio** — see section 2/3, `embed_audio` sits right next to `embed_vision`), so `AutoProcessor` likely also tries to build an audio sub-processor, which may need an optional dependency (`torchaudio`/`librosa`/similar) this environment doesn't have — completely unrelated to whether `transformers` itself is new enough. The cell below prints the REAL inner exception (not the truncated one-liner Jupyter shows by default) so that's diagnosable, and falls back to `AutoTokenizer` alone — `resolve_image_token_id` only ever reads `getattr(processor, "tokenizer", processor)`, so a bare tokenizer is a fully valid substitute for what THIS check needs, even if the full audio-capable processor won't load.

In [27]:
try:
    processor = AutoProcessor.from_pretrained(MODEL_ID)  # small config/tokenizer files only, no model weights
    print(f"Loaded full AutoProcessor: {type(processor).__name__}")
except Exception as e:
    # Print the WHOLE cause chain — the top-level ModuleNotFoundError from
    # transformers' lazy loader is a WRAPPER; the actual reason (missing
    # optional audio dependency, genuine version gap, or something else
    # entirely) is further down and Jupyter's default traceback view
    # truncates long ones before you get to it.
    print(f"AutoProcessor.from_pretrained failed: {type(e).__name__}: {e}\n")
    cause = e.__cause__
    depth = 1
    while cause is not None:
        print(f"  caused by [{depth}] {type(cause).__name__}: {cause}")
        cause = cause.__cause__
        depth += 1
    print("\nFalling back to AutoTokenizer alone — sufficient for this section, see the "
          "markdown above for why.")
    from transformers import AutoTokenizer
    processor = AutoTokenizer.from_pretrained(MODEL_ID)
    print(f"Loaded: {type(processor).__name__}")

image_token_id = resolve_image_token_id(model, processor)
tokenizer = getattr(processor, "tokenizer", processor)
print(f"\nimage_token_id = {image_token_id}")
print(f"decodes to: {tokenizer.convert_ids_to_tokens([image_token_id])}")

AutoProcessor.from_pretrained failed: ModuleNotFoundError: Could not import module 'Gemma4UnifiedProcessor'. Are this object's requirements defined correctly?

  caused by [1] ModuleNotFoundError: No module named 'torchvision'

Falling back to AutoTokenizer alone — sufficient for this section, see the markdown above for why.
Loaded: GemmaTokenizer

image_token_id = 258880
decodes to: ['<|image|>']


## 6. Parameter-count summary (still meta-tensor cheap — no real weights needed)

What `assert_only_language_trainable`'s `trainable_pct` will report once LoRA is actually attached, computed here from shapes alone as a preview.

In [28]:
total = param_count(model)
decoder = param_count(lm)
print(f"Total model params:        {total:,}")
print(f"Language decoder params:   {decoder:,}  ({100 * decoder / total:.1f}%)")
print(f"Vision + projector params: {total - decoder:,}  ({100 * (total - decoder) / total:.1f}%)  — stays fully frozen")
print(f"\nLinear layers targeted for a LoRA adapter: {len(targets)} "
      f"(every Linear inside the {decoder:,}-param decoder, none outside it)")

Total model params:        12,966,363,136
Language decoder params:   11,907,350,272  (91.8%)
Vision + projector params: 1,059,012,864  (8.2%)  — stays fully frozen

Linear layers targeted for a LoRA adapter: 328 (every Linear inside the 11,907,350,272-param decoder, none outside it)


## 7. Optional — the one check that needs REAL weights, not meta ones

This section actually loads the checkpoint (GPU or CPU, several minutes / needs real RAM) and re-runs `assert_only_language_trainable` — the SAME function `train_lora.py`'s `main()` calls — against the real `peft`-wrapped model, which is the closest this notebook gets to literally re-running the trainer's own setup path.

Built with `use_dora=False` (**plain LoRA**, matching `train_lora.py`'s actual default — `--use_dora` is opt-in there and not what this project trains with; see the module's docstring for why: vLLM can't serve a DoRA adapter selectively). Meta tensors would work fine for plain LoRA's own init (it needs no data from the base weights), but this cell loads real weights anyway so it exercises the exact same code path `train_lora.py --dataset_dir ... --output_dir ...` runs, not just the parts that happen to tolerate a meta tensor.

**Skip this cell** unless you specifically want to confirm the trainer's setup path runs clean on this checkpoint; sections 1-6 already verify everything about the layer *selection* logic without needing it.

In [29]:
RUN_REAL_WEIGHT_CHECK = False  # flip to True deliberately

if RUN_REAL_WEIGHT_CHECK:
    from peft import LoraConfig, get_peft_model

    real_model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, dtype=torch.bfloat16, device_map="cpu"  # cpu: this cell is about correctness, not speed
    )
    for p in real_model.parameters():
        p.requires_grad = False

    real_lm, real_prefix = language_model_of(real_model)
    real_targets = find_target_modules(real_model, real_prefix)

    peft_model = get_peft_model(real_model, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=real_targets, bias="none", task_type="CAUSAL_LM",
        use_dora=False,  # plain LoRA — matches train_lora.py's actual default
    ))
    stats = assert_only_language_trainable(peft_model, real_prefix)
    print("assert_only_language_trainable passed on the REAL, peft-wrapped model (plain LoRA):")
    print(stats)
else:
    print("Skipped (RUN_REAL_WEIGHT_CHECK = False). Sections 1-6 already verify layer selection "
          "without needing the real weights.")

Skipped (RUN_REAL_WEIGHT_CHECK = False). Sections 1-6 already verify layer selection without needing the real weights.
